In [40]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [41]:
url = "https://www.ccny.cuny.edu/registrar/fall"

response = requests.get(url)

print("Status code:", response.status_code)
print(response.text[:500])

Status code: 403
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src &#39;none&#39;; script-src &#39;nonce-QvJFHIfjWuLI6kC52dpN0b&#39; &#39;unsafe-eval&#39; https://challenges.cloudflare.com; script-s


In [42]:
with open("fall2021.html", "r") as file:
    html = file.read()

soup = BeautifulSoup(html, "html.parser")

In [43]:
rows = soup.find_all("tr")

print("Number of rows:", len(rows))

for row in rows:
    cells = row.find_all(["th", "td"])
    values = [cell.get_text(strip=True) for cell in cells]
    print(values)

Number of rows: 37
['DATES', 'DAYS', 'TEXT']
['August 01, 2021', 'Sunday', 'Application for degree for January and February 2022 begins']
['August 18, 2021', 'Wednesday', 'Last day to apply for Study Abroad']
['August 24, 2021', 'Tuesday', 'Last day of Registration; Last day to file ePermit for the Fall 2021; Last day to drop classes for 100% tuition refund;']
['August 25, 2021', 'Wednesday', 'Start of Fall Term; Classes begin; Initial Registration Appeals begin;']
['August 25 - 31, 2021', 'Wednesday - Tuesday', 'Change of program period; late fees apply']
['August 26, 2021', 'Thursday', 'Last day for Independent Study']
['August 28, 2021', 'Saturday', 'First day of Saturday Classes']
['August 31, 2021', 'Tuesday', "Last day to add a class to an existing enrollment; Last day for 75% tuition refund; Financial Aid Certification Enrollment Status date; Last day to apply for Audit option; Last day to drop without the grade of 'WD'; Initial Registration Appeals end;"]
['September 01, 2021',

In [44]:
data = []

for row in soup.find_all("tr")[1:]:
    cells = row.find_all("td")
    
    if len(cells) == 3:
        date_text = cells[0].get_text(strip=True)
        dow = cells[1].get_text(strip=True)
        text = cells[2].get_text(" ", strip=True)

        data.append({
            "date": date_text,
            "dow": dow,
            "text": text
        })

print(data)

[{'date': 'August 01, 2021', 'dow': 'Sunday', 'text': 'Application for degree for January and February 2022 begins'}, {'date': 'August 18, 2021', 'dow': 'Wednesday', 'text': 'Last day to apply for Study Abroad'}, {'date': 'August 24, 2021', 'dow': 'Tuesday', 'text': 'Last day of Registration; Last day to file ePermit for the Fall 2021; Last day to drop classes for 100% tuition refund;'}, {'date': 'August 25, 2021', 'dow': 'Wednesday', 'text': 'Start of Fall Term; Classes begin; Initial Registration Appeals begin;'}, {'date': 'August 25 - 31, 2021', 'dow': 'Wednesday - Tuesday', 'text': 'Change of program period; late fees apply'}, {'date': 'August 26, 2021', 'dow': 'Thursday', 'text': 'Last day for Independent Study'}, {'date': 'August 28, 2021', 'dow': 'Saturday', 'text': 'First day of Saturday Classes'}, {'date': 'August 31, 2021', 'dow': 'Tuesday', 'text': "Last day to add a class to an existing enrollment; Last day for 75% tuition refund; Financial Aid Certification Enrollment Stat

In [45]:
df = pd.DataFrame(data)

def convert_date(date_text):
    # If the date is a range such as "August 25 - 31, 2021",
    # use the first date in the range.
    if " - " in date_text:
        month_day, rest = date_text.split(" - ", 1)
        year = rest.split(",")[-1].strip()
        date_text = f"{month_day}, {year}"

    return pd.to_datetime(date_text)

df["date"] = df["date"].apply(convert_date)

df = df.set_index("date")

df

,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday - Tuesday,Change of program period; late fees apply
2021-08-26,Thursday,Last day for Independent Study
2021-08-28,Saturday,First day of Saturday Classes
2021-08-31,Tuesday,Last day to add a class to an existing enrollm...
2021-09-01,Wednesday,Verification of Enrollment rosters available t...


In [46]:
print(df)
print()
print("Number of rows:", len(df))
print()
print(df.index)
print()
print(df.dtypes)

                             dow  \
date                               
2021-08-01                Sunday   
2021-08-18             Wednesday   
2021-08-24               Tuesday   
2021-08-25             Wednesday   
2021-08-25   Wednesday - Tuesday   
2021-08-26              Thursday   
2021-08-28              Saturday   
2021-08-31               Tuesday   
2021-09-01             Wednesday   
2021-09-03    Friday - Wednesday   
2021-09-06                Monday   
2021-09-09              Thursday   
2021-09-14               Tuesday   
2021-09-15             Wednesday   
2021-09-15  Wednesday - Thursday   
2021-09-23              Thursday   
2021-09-24                Friday   
2021-10-01                Friday   
2021-10-08                Friday   
2021-10-11                Monday   
2021-11-01                Monday   
2021-11-02               Tuesday   
2021-11-04              Thursday   
2021-11-06              Saturday   
2021-11-23               Tuesday   
2021-11-25     Thursday - Su

In [47]:
print("Number of rows:", len(df))
print("First date:", df.index.min())
print("Last date:", df.index.max())
print("Columns:", df.columns.tolist())

Number of rows: 36
First date: 2021-08-01 00:00:00
Last date: 2022-01-01 00:00:00
Columns: ['dow', 'text']


## Final Result

The CCNY Fall 2021 calendar was converted into a pandas DataFrame.

- The DataFrame index is stored as a datetime type.
- The `dow` column contains the day of the week.
- The `text` column contains the calendar event description.
- The final dataset contains 36 calendar entries.

The original CCNY webpage returned a 403 response because of Cloudflare protection, so a local HTML copy of the Fall 2021 calendar was used for parsing with BeautifulSoup.